# Parity-Net Colab Runner
This notebook clones the GitHub repo, trains the mean-field-style residual network from `MOTIVATION.md`, saves checkpoints to Google Drive, and runs PCA rank-reduction analysis.
Before running, set `GITHUB_REPO_URL` to the URL of your pushed repo.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
GITHUB_REPO_URL = "https://github.com/labofdoubt/feature-learning-parity-task.git"
REPO_DIR = "/content/feature-learning-parity-task"

!rm -rf "$REPO_DIR"
!git clone "$GITHUB_REPO_URL"
%cd "$REPO_DIR"
!pip install -e .


In [ ]:
DRIVE_RUN_DIR = "/content/drive/MyDrive/ml_projects_new/parity_runs/example_run_8"

## Create Configs
`barrier_c` is in the training config because it enters as a loss regularizer. If left as `None`, the training code uses `7 / N`.



In [ ]:
from pathlib import Path
import yaml

N = 512

config = {
    "model": {
        "input_dim": 32,
        "relevant_dim": 16,
        "N": N,
        "L": 4,
        "activation": "relu",
        "use_readout_barrier": False,
        "embedding_weight_variance": 1.0/32,
        "hidden_weight_variance": 1.0/N,
        "readout_weight_variance": 1.0/N,
        "bias": False,
    },
    "training": {
        "num_steps": 10_000,
        "test_samples": 20_000,
        "batch_size": 256,
        "seed": 0,
        "device": "cuda",
        "dtype": "float32",
        "log_every": 100,
        "checkpoint_every": 1_000,
        "output_dir": DRIVE_RUN_DIR,
        "barrier_c": None,
        "barrier_lambda": 10.0,
        "optimizer": {
            "name": "adamw",
            "lr": 1e-4,
            "weight_decay": 0.0,
            "momentum": 0.0,
            "betas": [0.9, 0.999],
        },
    },
}
config_path = Path(DRIVE_RUN_DIR) / "config.yaml"
config_path.parent.mkdir(parents=True, exist_ok=True)
with config_path.open("w") as f:
    yaml.safe_dump(config, f, sort_keys=False)
config_path




PosixPath('/content/drive/MyDrive/ml_projects_new/parity_runs/example_run_7/config.yaml')

## Train

Training samples a fresh random batch at every optimizer step. `num_steps` is the number of weight updates.
This runs the repo command and saves intermediate checkpoints to Google Drive.



In [65]:
!parity-train --config "$config_path"


{'step': 100, 'elapsed_seconds': 0.550648922999244, 'train_mse': 0.8674699068069458, 'barrier': 0.0, 'loss': 0.8674699068069458, 'test_mse': 0.8639352321624756, 'test_mse_d2': 0.714694619178772, 'test_mse_d4': 1.0313268899917603, 'test_mse_d8': 1.0344973802566528, 'test_mse_d16': 1.0471680164337158}
{'step': 200, 'elapsed_seconds': 0.8978171489998203, 'train_mse': 0.6011594533920288, 'barrier': 0.0, 'loss': 0.6011594533920288, 'test_mse': 0.5924187898635864, 'test_mse_d2': 0.203122079372406, 'test_mse_d4': 1.036129117012024, 'test_mse_d8': 1.0391463041305542, 'test_mse_d16': 1.0384962558746338}
{'step': 300, 'elapsed_seconds': 1.2342906939993554, 'train_mse': 0.4894152581691742, 'barrier': 0.0, 'loss': 0.4894152581691742, 'test_mse': 0.4960329532623291, 'test_mse_d2': 0.04017750918865204, 'test_mse_d4': 1.0039751529693604, 'test_mse_d8': 1.036925196647644, 'test_mse_d16': 1.0293229818344116}
{'step': 400, 'elapsed_seconds': 1.569215828999404, 'train_mse': 0.4790881872177124, 'barrier':

## Load Checkpoint and Inspect Weight Variance



In [67]:
import torch
import pandas as pd
from parity_net.checkpoint import load_checkpoint
from parity_net.train import resolve_device
checkpoint_path = Path(DRIVE_RUN_DIR) / "checkpoints" / "step_00010000.pt"
device = resolve_device(config["training"]["device"])
model, payload, _ = load_checkpoint(checkpoint_path, device)
weight_variances = model.weight_variances()
pd.DataFrame([
    {"layer": layer, "variance": variance}
    for layer, variance in weight_variances.items()
])

,layer,variance
0,embedding.weight,0.001953
1,blocks.0.linear.weight,0.002050
2,blocks.1.linear.weight,0.001988
3,blocks.2.linear.weight,0.001999
4,blocks.3.linear.weight,0.002019
5,readout.weight,0.001802


## Inspect Predictions on Test Samples

Run inference on a fresh held-out batch and compare model outputs with the true parity targets for a few samples.


In [68]:
from parity_net.data import make_dataset
from parity_net.train import resolve_dtype

num_inspect_samples = 5
inspect_batch_size = 32

dtype = resolve_dtype(config["training"]["dtype"])
inspect_data = make_dataset(
    inspect_batch_size,
    config["model"]["input_dim"],
    config["model"]["relevant_dim"],
    device,
    dtype,
)

model.eval()
with torch.no_grad():
    inspect_pred = model(inspect_data.x)

input_columns = [f"x{i}" for i in range(config["model"]["input_dim"])]
output_columns = [
    *[f"d2_{i}" for i in range(8)],
    *[f"d4_{i}" for i in range(4)],
    *[f"d8_{i}" for i in range(2)],
    "d16_0",
]

for sample_idx in range(num_inspect_samples):
    input_sequence = pd.DataFrame(
        [inspect_data.x[sample_idx].detach().cpu().numpy().astype(int)],
        columns=input_columns,
    )
    comparison = pd.DataFrame(
        {
            "target": inspect_data.y[sample_idx].detach().cpu().numpy(),
            "prediction": inspect_pred[sample_idx].detach().cpu().numpy(),
        },
        index=output_columns,
    )
    print(f"Sample {sample_idx}: input sequence")
    display(input_sequence)
    print(f"Sample {sample_idx}: targets vs predictions")
    display(comparison)


Sample 0: input sequence


,x0,x1,x2,x3,x4,x5,x6,x7,x8,x9,...,x22,x23,x24,x25,x26,x27,x28,x29,x30,x31
0,1,1,-1,1,-1,1,1,-1,1,-1,...,1,1,-1,-1,1,1,-1,-1,1,-1


Sample 0: targets vs predictions


,target,prediction
d2_0,1.0,1.000717
d2_1,-1.0,-1.024024
d2_2,-1.0,-0.976866
d2_3,-1.0,-0.991550
d2_4,-1.0,-0.993541
d2_5,1.0,0.976345
d2_6,-1.0,-0.993547
d2_7,-1.0,-1.010297
d4_0,-1.0,-1.012908
d4_1,1.0,0.995046


Sample 1: input sequence


...

,target,prediction
d2_0,-1.0,-0.975374
d2_1,1.0,1.006014
d2_2,-1.0,-0.990774
d2_3,-1.0,-1.000374
d2_4,-1.0,-0.999574
d2_5,1.0,0.996632
d2_6,-1.0,-1.005022
d2_7,-1.0,-1.035093
d4_0,-1.0,-1.025124
d4_1,1.0,1.015242


Sample 4: input sequence


,x0,x1,x2,x3,x4,x5,x6,x7,x8,x9,...,x22,x23,x24,x25,x26,x27,x28,x29,x30,x31
0,-1,-1,1,-1,1,-1,1,-1,-1,-1,...,-1,-1,1,1,-1,-1,1,-1,-1,1


Sample 4: targets vs predictions


,target,prediction
d2_0,1.0,1.028540
d2_1,-1.0,-1.038050
d2_2,-1.0,-1.018564
d2_3,-1.0,-0.992289
d2_4,1.0,1.030142
d2_5,-1.0,-1.028498
d2_6,-1.0,-0.981213
d2_7,-1.0,-1.029812
d4_0,-1.0,-1.010544
d4_1,1.0,0.994839


## Inspect Layerwise Activation Variance

Run inference on a fresh input batch and measure, for each sample, the activation variance across the hidden dimension after the embedding and after each residual block.


In [69]:
activation_batch_size = 256

activation_data = make_dataset(
    activation_batch_size,
    config["model"]["input_dim"],
    config["model"]["relevant_dim"],
    device,
    dtype,
)

model.eval()
with torch.no_grad():
    _, activations = model(activation_data.x, return_activations=True)

rows = []
for layer_idx, h in enumerate(activations):
    layer_name = "embedding" if layer_idx == 0 else f"block_{layer_idx}"
    sample_variances = h.detach().float().var(dim=1, unbiased=False)
    rows.append(
        {
            "layer_idx": layer_idx,
            "layer": layer_name,
            "mean_sample_variance": sample_variances.mean().item(),
            "std_sample_variance": sample_variances.std(unbiased=False).item(),
            "min_sample_variance": sample_variances.min().item(),
            "max_sample_variance": sample_variances.max().item(),
        }
    )

activation_variance_df = pd.DataFrame(rows)
display(activation_variance_df)


,layer_idx,layer,mean_sample_variance,std_sample_variance,min_sample_variance,max_sample_variance
0,0,embedding,0.062387,0.000153,0.061467,0.062500
1,1,block_1,0.108081,0.006907,0.091446,0.126596
2,2,block_2,0.175662,0.010985,0.146936,0.208626
3,3,block_3,0.502717,0.044751,0.380802,0.647935
4,4,block_4,1.722122,0.152128,1.318057,2.203554


## PCA Rank-Reduction Analysis
Set `INTERVENTION_LAYER_IDX` and `KEEP_PCS` below. Layer index `0` is the embedded residual stream before the first residual block; later indices are after residual blocks / before the corresponding next block.



In [ ]:
INTERVENTION_LAYER_IDX = 2
KEEP_PCS = 50
PCA_SAMPLES = config["training"]["test_samples"]
ANALYSIS_DIR = str(Path(DRIVE_RUN_DIR) / "analysis")
!parity-analyze   --checkpoint "$checkpoint_path"   --output-dir "$ANALYSIS_DIR"   --pca-samples "$PCA_SAMPLES"   --batch-size "{config['training']['batch_size']}"   --intervention-layer "$INTERVENTION_LAYER_IDX"   --keep-pcs "$KEEP_PCS"



## Results
The first table reports how many dimensions recover 90% and 99% of variance at each layer. The second reports MSE after the PCA intervention, including degree 2, 4, 8, and 16 parity groups.



In [ ]:
rank_df = pd.read_csv(Path(ANALYSIS_DIR) / "pca_rank_thresholds.csv")
intervention_df = pd.read_csv(Path(ANALYSIS_DIR) / "pca_intervention_mse.csv")
baseline_df = pd.read_csv(Path(ANALYSIS_DIR) / "baseline_mse.csv")
print("PCA rank thresholds")
display(rank_df)
print("Baseline MSE")
display(baseline_df)
print("PCA intervention MSE")
display(intervention_df[["intervention_layer", "keep_pcs", "mse_d2", "mse_d4", "mse_d8", "mse_d16", "mse_all"]])

